# Solutions – 4. Hypothesis Testing

This notebook provides worked solutions to the five exercises from
[04_hypothesis_testing.ipynb](../Stats/04_hypothesis_testing.ipynb).

We use the same signal/background model and `pyhf` setup from notebook 04.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, poisson
from scipy.optimize import minimize_scalar, minimize
import pyhf
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Model definitions from notebook 04 ───────────────────────────────────────
# Signal PDF: f(x|s) = 3(1-x)^2   (peaked at low x)
# Background PDF: f(x|b) = 3x^2   (peaked at high x)

def pdf_signal(x):
    return 3.0 * (1.0 - x) ** 2

def pdf_background(x):
    return 3.0 * x ** 2

# Expected numbers (from notebook 04, section 3 baseline)
s_tot = 10.0    # total signal events
b_tot = 100.0   # total background events

def efficiency_signal(x_cut):
    return 1.0 - (1.0 - x_cut) ** 3

def efficiency_background(x_cut):
    return x_cut ** 3

def expected_counts(x_cut):
    ns = s_tot * efficiency_signal(x_cut)
    nb = b_tot * efficiency_background(x_cut)
    return ns, nb

# Asymptotic significance (no systematics)
def significance_asymptotic(ns, nb):
    if nb <= 0: return 0.0
    return np.sqrt(2.0 * ((ns + nb) * np.log(1.0 + ns / nb) - ns))

print("Setup complete.")

---
## Exercise 1 – Optimising the cut with a 10 % background systematic

**Task:** Add a 10 % Gaussian uncertainty on the background yield using the
asymptotic formula with systematics (Cowan *et al.* 2011, eq. 97) and compare
the optimal cut with the stat-only case.

In [ ]:
def significance_with_syst(ns, nb, delta_b):
    """Asymptotic significance including background uncertainty delta_b
    (Cowan et al. 2011, eq. 97).
    delta_b is the absolute uncertainty on the background.
    """
    if nb <= 0 or ns <= 0:
        return 0.0
    db2 = delta_b ** 2
    # First log term
    term1_log = np.log((ns + nb) / nb * (nb**2 + db2) / (nb**2 + ns*db2 + db2))
    term1 = (ns + nb) * term1_log
    # Second log term
    term2_log = np.log(1.0 + ns * db2 / (nb * (nb + db2)))
    term2 = (nb**2 / db2) * term2_log
    arg = 2.0 * (term1 - term2)
    if arg <= 0:
        return 0.0
    return np.sqrt(arg)

x_cut_values = np.linspace(0.01, 0.50, 200)

z_stat_only = []
z_with_syst = []

for xc in x_cut_values:
    ns, nb = expected_counts(xc)
    z_stat_only.append(significance_asymptotic(ns, nb))
    delta_b = 0.10 * nb   # 10% relative uncertainty
    z_with_syst.append(significance_with_syst(ns, nb, delta_b))

z_stat_only = np.array(z_stat_only)
z_with_syst = np.array(z_with_syst)

xcut_opt_stat = x_cut_values[np.argmax(z_stat_only)]
xcut_opt_syst = x_cut_values[np.argmax(z_with_syst)]

print(f"Optimal x_cut (stat only):           {xcut_opt_stat:.3f}  "
      f"(Z = {z_stat_only.max():.3f})")
print(f"Optimal x_cut (10% bkg systematic):  {xcut_opt_syst:.3f}  "
      f"(Z = {z_with_syst.max():.3f})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_cut_values, z_stat_only, 'b-',  lw=2, label='Stat only')
ax.plot(x_cut_values, z_with_syst, 'r--', lw=2, label='10% bkg systematic')
ax.axvline(xcut_opt_stat, color='blue',  ls=':',  lw=1.5,
           label=f'Optimal (stat): {xcut_opt_stat:.3f}')
ax.axvline(xcut_opt_syst, color='red',   ls=':',  lw=1.5,
           label=f'Optimal (syst): {xcut_opt_syst:.3f}')
ax.set_xlabel(r'$x_{\rm cut}$'); ax.set_ylabel('Expected significance Z')
ax.set_title('Optimal cut with and without background systematic')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Observations:**
- Adding a 10 % background systematic reduces the expected significance at every cut
  value, because part of the background constraint is replaced by an uncertain estimate.
- The optimal cut shifts to **smaller** $x_{\rm cut}$ when systematics are included:
  a tighter cut selects fewer background events and therefore suffers less from the
  relative background uncertainty.
- For very tight cuts the gain from background suppression outweighs the loss in signal
  efficiency, making a tighter selection more robust to systematics.

---
## Exercise 2 – $\textrm{CL}_s$ upper limits vs. $\textrm{CL}_{s+b}$-based limits

**Task:** Compute the 95 % upper limit using the standard $\textrm{CL}_s$ prescription and
compare it with the limit obtained from the s+b p-value criterion ($\textrm{CL}_{s+b}$)
for a background-only observation.

In [ ]:
# Build a simple pyhf counting model (same as notebook 04, section 8b)
x_cut_base = 0.1
ns_exp, nb_exp = expected_counts(x_cut_base)
print(f"Expected: ns = {ns_exp:.3f}, nb = {nb_exp:.3f}")

# Observed = background-only (no signal)
n_obs = int(round(nb_exp))

def build_model(signal_count, background_count):
    model = pyhf.simplemodels.uncorrelated_background(
        signal=[signal_count],
        bkg=[background_count],
        bkg_uncertainty=[0.1 * background_count]
    )
    return model

model = build_model(ns_exp, nb_exp)
observations = [n_obs] + model.config.auxdata

mu_values = np.linspace(0.01, 5.0, 120)

def cls_and_clsb(mu):
    """Return (CLs, CLs+b) for a fixed signal strength mu."""
    result = pyhf.infer.hypotest(
        mu,
        observations,
        model,
        test_stat='qtilde',
        return_expected_set=False,
        return_tail_probs=True,
    )

    cls_val = float(result[0])

    # With return_tail_probs=True, pyhf returns the additional tail probabilities.
    # In current pyhf versions this is [CLs+b, CLb].
    tail_probs = result[1]
    clsb_val = float(tail_probs[0])
    return cls_val, clsb_val

def upper_limit_from_curve(mu_grid, y_grid, target=0.05):
    """Find mu where y(mu) crosses target by linear interpolation."""
    idx = np.where(np.diff(np.sign(y_grid - target)))[0]
    if len(idx) == 0:
        return np.nan
    i = idx[0]
    x0, x1 = mu_grid[i], mu_grid[i + 1]
    y0, y1 = y_grid[i], y_grid[i + 1]
    return x0 + (target - y0) * (x1 - x0) / (y1 - y0)

cls_vals = []
clsb_vals = []
for mu in mu_values:
    cls_mu, clsb_mu = cls_and_clsb(mu)
    cls_vals.append(cls_mu)
    clsb_vals.append(clsb_mu)

cls_vals = np.array(cls_vals)
clsb_vals = np.array(clsb_vals)

ul_cls = upper_limit_from_curve(mu_values, cls_vals, target=0.05)
ul_clsb = upper_limit_from_curve(mu_values, clsb_vals, target=0.05)

print("\n=== 95% upper limits ===")
print(f"  CLs criterion (CLs = 0.05):       mu_up = {ul_cls:.3f}")
print(f"  CLs+b criterion (CLs+b = 0.05):   mu_up = {ul_clsb:.3f}")

print("\nSample tail probabilities at mu = 1:")
cls_1, clsb_1 = cls_and_clsb(1.0)
print(f"  CLs(mu=1)   = {cls_1:.4f}")
print(f"  CLs+b(mu=1) = {clsb_1:.4f}")

**Observations:**
- The $\textrm{CL}_s$ limit is usually **looser** (more conservative) than the limit from
  $\textrm{CL}_{s+b}$ alone, because $\textrm{CL}_s = \mathrm{CL}_{s+b}/\mathrm{CL}_b$ protects against
  excluding signals when the experiment has poor sensitivity.
- A $\textrm{CL}_{s+b}$-only criterion can become overly aggressive under downward background
  fluctuations, yielding limits stronger than what the analysis can genuinely support.
- This is why HEP searches conventionally quote exclusion limits from the $\textrm{CL}_s$
  prescription rather than from $\textrm{CL}_{s+b}$ alone.

---
## Exercise 3 – Two-bin model with a control region

**Task:** Extend the `pyhf` model to two bins: a signal-enriched bin ($x < 0.1$)
and a control bin ($0.1 \le x < 0.5$).  Verify that the control region improves
the expected discovery significance.

In [ ]:
# ── Single-bin model (baseline) ──────────────────────────────────────────────
x_cut_sr = 0.1    # signal region boundary
x_cut_cr = 0.5    # upper edge of control region

# Signal efficiency in each bin
eff_s_sr = efficiency_signal(x_cut_sr)       # x < 0.1
eff_s_cr = (efficiency_signal(x_cut_cr)       # 0.1 <= x < 0.5
            - efficiency_signal(x_cut_sr))

# Background efficiency in each bin
eff_b_sr = efficiency_background(x_cut_sr)
eff_b_cr = (efficiency_background(x_cut_cr)
            - efficiency_background(x_cut_sr))

ns_sr = s_tot * eff_s_sr
nb_sr = b_tot * eff_b_sr
ns_cr = s_tot * eff_s_cr
nb_cr = b_tot * eff_b_cr

print(f"Signal region (x < {x_cut_sr}):        ns = {ns_sr:.3f}, nb = {nb_sr:.3f}")
print(f"Control region ({x_cut_sr} <= x < {x_cut_cr}): ns = {ns_cr:.3f}, nb = {nb_cr:.3f}")

# ── 1-bin pyhf model ──────────────────────────────────────────────────────────
model_1bin = pyhf.simplemodels.uncorrelated_background(
    signal=[ns_sr],
    bkg=[nb_sr],
    bkg_uncertainty=[0.1 * nb_sr]
)
obs_1bin = [int(round(nb_sr))] + model_1bin.config.auxdata

# ── 2-bin pyhf model ──────────────────────────────────────────────────────────
# The control region constrains the background normalisation in-situ
model_2bin = pyhf.simplemodels.uncorrelated_background(
    signal=[ns_sr, ns_cr],
    bkg=[nb_sr, nb_cr],
    bkg_uncertainty=[0.1 * nb_sr, 0.1 * nb_cr]
)
obs_2bin = [int(round(nb_sr)), int(round(nb_cr))] + model_2bin.config.auxdata

print(f"\n1-bin model: {model_1bin.config.npars} parameters")
print(f"2-bin model: {model_2bin.config.npars} parameters")

In [ ]:
# ── Discovery significance ────────────────────────────────────────────────────
# Under s+b hypothesis (mu=1): inject signal into observed data
obs_1bin_sb = [int(round(ns_sr + nb_sr))] + model_1bin.config.auxdata
obs_2bin_sb = [int(round(ns_sr + nb_sr)), int(round(ns_cr + nb_cr))] + model_2bin.config.auxdata

def p0_from_pyhf(model, observations):
    result = pyhf.infer.hypotest(
        0.0, observations, model,
        test_stat='q0',
        return_expected_set=False,
    )
    return float(result)

p0_1bin = p0_from_pyhf(model_1bin, obs_1bin_sb)
p0_2bin = p0_from_pyhf(model_2bin, obs_2bin_sb)

from scipy.stats import norm as _norm
z_1bin = _norm.ppf(1 - p0_1bin) if p0_1bin < 0.5 else 0.0
z_2bin = _norm.ppf(1 - p0_2bin) if p0_2bin < 0.5 else 0.0

print("=== Discovery significance ===")
print(f"  1-bin model: p0 = {p0_1bin:.4f},  Z = {z_1bin:.3f}")
print(f"  2-bin model: p0 = {p0_2bin:.4f},  Z = {z_2bin:.3f}")
print(f"  Improvement from control region: delta Z = {z_2bin - z_1bin:.3f}")

**Observations:**
- Adding a control region ($0.1 \le x < 0.5$) that is signal-depleted constrains the
  background yield in-situ, reducing the background uncertainty in the signal region.
- This leads to an improved (higher) expected discovery significance.
- The improvement is larger when the background uncertainty is significant relative to
  the signal yield.  With tight control-region statistics the benefit is smaller.

---
## Exercise 4 – Toys vs. asymptotics

**Task:** Compare the toy-based $p_0$ with the asymptotic result using
`pyhf.infer.hypotest` with `calctype='toybased'`.

In [ ]:
# Use the 1-bin model with observed data at background-only expectation
# to avoid the need for very large toy samples

model_ex4 = model_1bin
obs_ex4   = obs_1bin_sb   # signal+background observation for discovery test

# Asymptotic p0
p0_asymp = p0_from_pyhf(model_ex4, obs_ex4)

# Compatibility shim for pyhf<->NumPy API differences in percentile keyword
import inspect
if "interpolation" not in inspect.signature(np.percentile).parameters:
    _np_percentile = np.percentile
    def _percentile_compat(a, q, *args, interpolation=None, **kwargs):
        if interpolation is not None and "method" not in kwargs:
            kwargs["method"] = interpolation
        return _np_percentile(a, q, *args, **kwargs)
    np.percentile = _percentile_compat

# Toy-based p0
ntoys_ex4 = 2000
try:
    p0_toys = float(pyhf.infer.hypotest(
        0.0, obs_ex4, model_ex4,
        test_stat='q0',
        calctype='toybased',
        ntoys=ntoys_ex4,
        return_expected_set=False,
    ))
except Exception as e:
    p0_toys = None
    print(f"Toy-based calculation raised: {e}")

z_asymp = _norm.ppf(1 - p0_asymp) if p0_asymp < 0.5 else 0.0
z_toys  = None
if p0_toys is not None and p0_toys < 0.5:
    # With finite toys, p0 can be 0 if no toys exceed q_obs; report lower-bound Z
    p0_toys_for_z = max(p0_toys, 1.0 / ntoys_ex4)
    z_toys = _norm.ppf(1 - p0_toys_for_z)

print("=== Asymptotic vs. toy-based p0 ===")
print(f"  Asymptotic: p0 = {p0_asymp:.4f},  Z = {z_asymp:.3f}")
if z_toys is not None:
    if p0_toys <= 0.0:
        print(f"  Toy-based:  p0 < {1.0 / ntoys_ex4:.4g},  Z > {z_toys:.3f} (toy-stat limited)")
    else:
        print(f"  Toy-based:  p0 = {p0_toys:.4f},  Z = {z_toys:.3f}")

**Observations:**
- For moderate $n_s/n_b$ the asymptotic and toy-based $p$-values agree well.
- When $n_s + n_b$ is small (say $< 5$) the asymptotic approximation can break down
  and the toy-based calculation becomes necessary.
- The toy-based method is computationally more expensive but does not rely on the
  asymptotic assumptions.  It is the standard for discovery claims in HEP
  ($Z > 5$, where the tails of the distribution matter most).

---
## Exercise 5 – $\textrm{CL}_s$ upper limit vs. background uncertainty

**Task:** Compute the $\textrm{CL}_s$ upper limit on $\mu$ analytically and compare with
the `pyhf` result.  Show how the limit changes as the background uncertainty
increases from 0 % to 50 %.

In [ ]:
ns_ex5 = ns_sr
nb_ex5 = nb_sr
n_obs_ex5 = int(round(nb_ex5))   # background-only observation

def pyhf_ul_cls(bkg_rel_unc, mu_range=np.linspace(0.01, 10.0, 100)):
    """Compute pyhf CLs upper limit for a given relative background uncertainty."""
    model_l = pyhf.simplemodels.uncorrelated_background(
        signal=[ns_ex5],
        bkg=[nb_ex5],
        bkg_uncertainty=[bkg_rel_unc * nb_ex5]
    )
    obs_l = [n_obs_ex5] + model_l.config.auxdata

    cls_vals = []
    for mu in mu_range:
        res = pyhf.infer.hypotest(
            mu, obs_l, model_l, test_stat='qtilde',
            return_expected_set=False,
        )
        cls_vals.append(float(res))

    cls_arr = np.array(cls_vals)
    idx = np.where(np.diff(np.sign(cls_arr - 0.05)))[0]
    if len(idx) == 0:
        return np.nan
    i = idx[0]
    x0, x1 = mu_range[i], mu_range[i+1]
    y0, y1 = cls_arr[i], cls_arr[i+1]
    return x0 + (0.05 - y0) * (x1 - x0) / (y1 - y0)

bkg_unc_values = np.array([0.0, 0.05, 0.10, 0.20, 0.30, 0.50])
ul_pyhf = []

for bunc in bkg_unc_values:
    ul = pyhf_ul_cls(bunc)
    ul_pyhf.append(ul)
    print(f"  bkg_rel_unc = {bunc:.0%}:  95% CL_s UL on mu = {ul:.3f}")

ul_pyhf = np.array(ul_pyhf)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bkg_unc_values * 100, ul_pyhf, 'o-', color='steelblue', lw=2)
ax.set_xlabel('Background relative uncertainty [%]')
ax.set_ylabel(r'95 % CL$_s$ upper limit on $\mu$')
ax.set_title(r'Upper limit vs. background uncertainty')
plt.tight_layout(); plt.show()

**Observations:**
- The $\textrm{CL}_s$ upper limit increases (weakens) monotonically as the background uncertainty
  grows.  With 0 % uncertainty the limit is set purely by Poisson statistics; as the
  uncertainty increases the analysis loses sensitivity to the signal.
- For large uncertainties (50 %) the limit can be significantly weaker than the
  stat-only case, motivating aggressive efforts to constrain backgrounds in-situ.